In [ ]:
# /// script
# requires-python = ">=3.13"
# dependencies = [
#     "boto3",
#     "httpx",
#     "pystac",
#     "pystac-client",
#     "smart-open",
#     "stac-pydantic",
# ]
# ///

# Update Providers for all collections in the MAAP STAC

We worked through all of the collections in the MAAP STAC and worked out the producer/processor/host/licensor for each one ([spreadsheet](https://docs.google.com/spreadsheets/d/1XTDMPkD9IWFRgVZn4OvKeCGHdZTFnPaGFfp2qrQrNgE/edit?gid=0#gid=0)).

This notebook takes the assignments for each role and uses the entry in the [STAC Provider Definitions library](https://raw.githubusercontent.com/MAAP-Project/stac-providers/refs/heads/feat/v1/providers.json) to construct the Providers for each collection.

In [2]:
import json
import os
from typing import List, Tuple

import boto3
import smart_open
import stac_pydantic
from pystac import Provider, ProviderRole
from pystac_client import Client

provider_libary_url = "https://raw.githubusercontent.com/MAAP-Project/stac-providers/refs/heads/feat/v1/providers.json"

stage = os.getenv("STAGE", "test")

if stage == "prod":
    STAC_LOADER_SNS_TOPIC_ARN = "arn:aws:sns:us-west-2:916098889494:MAAP-STAC-dev-pgSTAC-stacitemloaderTopicD9D06088-m0iaNFNtnXUP"
else:
    STAC_LOADER_SNS_TOPIC_ARN = "arn:aws:sns:us-west-2:916098889494:MAAP-STAC-test-pgSTAC-stacitemloaderTopicD9D06088-LutBraKgk6sT"


client = Client.open("https://stac.maap-project.org")

# fetch all collections from the published catalog
collections = list(client.get_all_collections())

test


## Load the STAC Provider Definitions JSON

In [2]:
with smart_open.open(provider_libary_url) as f:
    provider_defs = json.load(f)

provider_lib = {provider["name"]: provider for provider in provider_defs}
provider_lib

{'ASF': {'name': 'ASF',
  'description': 'Alaska Satellite Facility',
  'url': 'https://www.asf.alaska.edu/'},
 'DE/DLR': {'name': 'DE/DLR',
  'description': 'German Aerospace Center (DLR)',
  'url': 'https://www.dlr.de'},
 'DOD/NGA': {'name': 'DOD/NGA',
  'description': 'National Geospatial-Intelligence Agency, U. S. Department of Defense',
  'url': 'https://www.nga.mil/Pages/Default.aspx'},
 'DOI/USGS/EROS': {'name': 'DOI/USGS/EROS',
  'description': 'Earth Resources Observation and Science Center, U.S. Geological Survey, U.S. Department of the Interior',
  'url': 'https://www.usgs.gov/centers/eros'},
 'ESA/CCI': {'name': 'ESA/CCI',
  'description': 'Climate Change Initiative, European Space Agency',
  'url': 'https://climate.esa.int/'},
 'ESA/ESRIN': {'name': 'ESA/ESRIN',
  'description': 'ESRIN Earth Observation, European Space Agency',
  'url': 'https://www.esa.int/About_Us/ESRIN/About_Earth_Observation'},
 'GMW': {'name': 'GMW',
  'description': 'Global Mangrove Watch',
  'url': 

## Define the providers entries for each collection

In [9]:
def set_providers(config: List[Tuple[str, List[str]]]) -> List[Provider]:
    """re-usable function to return a list of providers given provider names/roles"""
    return [
        Provider(roles=[ProviderRole(role) for role in roles], **provider_lib[name])
        for name, roles in config
    ]

List the roles for each provider that appears in a collection's row in the [spreadsheet](https://docs.google.com/spreadsheets/d/1XTDMPkD9IWFRgVZn4OvKeCGHdZTFnPaGFfp2qrQrNgE/edit?gid=0#gid=0).

In [5]:
provider_configs = {
    "ABoVE_UAVSAR_PALSAR": [
        ("NASA/ABoVE", ["producer", "processor", "licensor"]),
        ("NASA/MAAP", ["host"]),
    ],
    "AFRISAR_DLR": [
        ("DE/DLR", ["producer", "processor", "licensor"]),
        ("ESA/ESRIN", ["host"]),
    ],
    "AFRISAR_DLR2": [
        ("DE/DLR", ["producer", "processor", "licensor"]),
        ("ESA/ESRIN", ["host"]),
    ],
    "AfriSAR_UAVSAR_Coreg_SLC": [
        ("NASA/JPL", ["producer", "processor", "licensor"]),
        ("ORNL_DAAC", ["processor"]),
        ("NASA/MAAP", ["host"]),
    ],
    "AfriSAR_UAVSAR_Geocoded_Covariance": [
        ("NASA/JPL", ["producer", "processor", "licensor"]),
        ("ORNL_DAAC", ["processor"]),
        ("NASA/MAAP", ["host"]),
    ],
    "AfriSAR_UAVSAR_Geocoded_SLC": [
        ("NASA/JPL", ["producer", "processor", "licensor"]),
        ("ORNL_DAAC", ["processor"]),
        ("NASA/MAAP", ["host"]),
    ],
    "AfriSAR_UAVSAR_KZ": [
        ("NASA/JPL", ["producer", "processor", "licensor"]),
        ("ORNL_DAAC", ["processor"]),
        ("NASA/MAAP", ["host"]),
    ],
    "AfriSAR_UAVSAR_Normalization_Area": [
        ("NASA/JPL", ["producer", "processor", "licensor"]),
        ("ORNL_DAAC", ["processor"]),
        ("NASA/MAAP", ["host"]),
    ],
    "AfriSAR_UAVSAR_Ungeocoded_Covariance": [
        ("NASA/JPL", ["producer", "processor", "licensor"]),
        ("ORNL_DAAC", ["processor"]),
        ("NASA/MAAP", ["host"]),
    ],
    "BIOSAR1": [
        ("DE/DLR", ["producer", "processor"]),
        ("ESA/ESRIN", ["producer", "processor", "licensor"]),
        ("NASA/MAAP", ["host"]),
    ],
    "ESACCI_Biomass_L4_AGB_V4_100m": [
        ("ESA/CCI", ["producer", "processor"]),
        ("ESA/ESRIN", ["licensor"]),
        ("NASA/MAAP", ["processor", "host"]),
    ],
    "GEDI_CalVal_Field_Data": [
        ("NASA/GSFC/GEDI", ["producer", "processor", "licensor"]),
        ("NASA/MAAP", ["host"]),
    ],
    "GEDI_CalVal_Lidar_COPC": [
        ("NASA/GSFC/GEDI", ["producer", "processor", "licensor"]),
        ("NASA/MAAP", ["host"]),
    ],
    "GEDI_CalVal_Lidar_Data": [
        ("NASA/GSFC/GEDI", ["producer", "processor", "licensor"]),
        ("NASA/MAAP", ["host"]),
    ],
    "GEDI_CalVal_Lidar_Data_Compressed": [
        ("NASA/GSFC/GEDI", ["producer", "processor", "licensor"]),
        ("NASA/MAAP", ["host"]),
    ],
    "glad-glclu2020-change-v2": [
        ("UMD/GLAD", ["producer", "processor", "licensor"]),
        ("NASA/MAAP", ["processor", "host"]),
    ],
    "glad-glclu2020-v2": [
        ("UMD/GLAD", ["producer", "processor", "licensor"]),
        ("NASA/MAAP", ["processor", "host"]),
    ],
    "glad-global-forest-change-1.11": [
        ("UMD/GLAD", ["producer", "processor", "licensor"]),
        ("NASA/MAAP", ["processor", "host"]),
    ],
    "Global_Forest_Change_2000-2017": [
        ("UMD/GLAD", ["producer", "processor", "licensor"]),
        ("NASA/MAAP", ["processor", "host"]),
    ],
    "global-mangrove-watch-3.0": [
        ("GMW", ["producer", "processor", "licensor"]),
        ("NASA/MAAP", ["processor", "host"]),
    ],
    "Global_PALSAR2_PALSAR_FNF": [
        ("JP/JAXA/EORC", ["producer", "processor", "licensor"]),
        ("NASA/MAAP", ["host"]),
    ],
    "GlobCover_05_06": [
        ("ESA/ESRIN", ["producer", "processor", "licensor"]),
        ("UCLouvain", ["producer", "processor", "licensor"]),
        ("NASA/MAAP", ["host"]),
    ],
    "GlobCover_09": [
        ("ESA/ESRIN", ["producer", "processor", "licensor"]),
        ("UCLouvain", ["producer", "processor", "licensor"]),
        ("NASA/MAAP", ["host"]),
    ],
    "icesat2-boreal": [
        ("NASA/ABoVE", ["producer", "licensor"]),
        ("ESA/ESRIN", ["producer", "licensor"]),
        ("NASA/MAAP", ["processor", "host"]),
    ],
    "ICESat2_Boreal_AGB_tindex_average": [
        ("NASA/ABoVE", ["producer", "licensor"]),
        ("ESA/ESRIN", ["producer", "licensor"]),
        ("NASA/MAAP", ["processor", "host"]),
    ],
    "icesat2-boreal-v2.1-agb": [
        ("NASA/ABoVE", ["producer", "licensor"]),
        ("ESA/ESRIN", ["producer", "licensor"]),
        ("NASA/MAAP", ["processor", "host"]),
    ],
    "icesat2-boreal-v2.1-ht": [
        ("NASA/ABoVE", ["producer", "licensor"]),
        ("ESA/ESRIN", ["producer", "licensor"]),
        ("NASA/MAAP", ["processor", "host"]),
    ],
    "icesat2-boreal-v3.0-agb": [
        ("NASA/ABoVE", ["producer", "licensor"]),
        ("ESA/ESRIN", ["producer", "licensor"]),
        ("NASA/MAAP", ["processor", "host"]),
    ],
    "icesat2-boreal-v3.0-ht": [
        ("NASA/ABoVE", ["producer", "licensor"]),
        ("ESA/ESRIN", ["producer", "licensor"]),
        ("NASA/MAAP", ["processor", "host"]),
    ],
    "Landsat8_SurfaceReflectance": [
        ("DOI/USGS/EROS", ["producer", "processor", "licensor"]),
        ("NASA/MAAP", ["producer", "processor", "host"]),
    ],
    "NCEO_Africa_AGB_100m_2017": [
        ("UK/NERC/NCEO", ["producer"]),
        ("LEICESTER-U/ASTRO", ["producer", "processor", "licensor"]),
        ("NASA/MAAP", ["host"]),
    ],
    "nisar-sim": [
        ("NASA/JPL", ["producer", "processor", "licensor"]),
        ("ASF", ["host"]),
    ],
    "Paraguay_Country_Pilot": [
        ("USDA/FS/RMRS", ["producer", "processor", "licensor"]),
        ("INFONA", ["producer", "processor", "licensor"]),
        ("NASA/MAAP", ["processor", "host"]),
    ],
    "SRTMGL1_COD": [
        ("NASA/JPL/SRTM", ["producer", "processor", "licensor"]),
        ("DOD/NGA", ["producer", "licensor"]),
        ("DE/DLR", ["producer", "licensor"]),
        ("IT/ASI", ["producer", "licensor"]),
        ("NASA/MAAP", ["processor", "host"]),
    ],
}

Update the collections' providers entry and validate the collection data.

In [6]:
for collection in collections:
    config = provider_configs[collection.id]
    collection.providers = set_providers(config)
    collection.validate()
    collection_dict = collection.to_dict()
    _ = stac_pydantic.Collection(**collection_dict)
    # print(collection.id, "\n", json.dumps(collection_dict["providers"], indent=2), "\n", "-" * 80)

Post to the STAC Loader SNS topic to ingest to pgstac

In [8]:
# post collections to StacLoader
sns_client = boto3.client("sns")

for collection in collections:
    response = sns_client.publish(
        TopicArn=STAC_LOADER_SNS_TOPIC_ARN, Message=json.dumps(collection.to_dict())
    )
    print(collection.id, response["ResponseMetadata"]["HTTPStatusCode"])

ABoVE_UAVSAR_PALSAR 200
AFRISAR_DLR 200
AFRISAR_DLR2 200
AfriSAR_UAVSAR_Coreg_SLC 200
AfriSAR_UAVSAR_Geocoded_Covariance 200
AfriSAR_UAVSAR_Geocoded_SLC 200
AfriSAR_UAVSAR_KZ 200
AfriSAR_UAVSAR_Normalization_Area 200
AfriSAR_UAVSAR_Ungeocoded_Covariance 200
BIOSAR1 200
ESACCI_Biomass_L4_AGB_V4_100m 200
GEDI_CalVal_Field_Data 200
GEDI_CalVal_Lidar_COPC 200
GEDI_CalVal_Lidar_Data 200
GEDI_CalVal_Lidar_Data_Compressed 200
glad-glclu2020-change-v2 200
glad-glclu2020-v2 200
glad-global-forest-change-1.11 200
Global_Forest_Change_2000-2017 200
global-mangrove-watch-3.0 200
Global_PALSAR2_PALSAR_FNF 200
GlobCover_05_06 200
GlobCover_09 200
icesat2-boreal 200
ICESat2_Boreal_AGB_tindex_average 200
icesat2-boreal-v2.1-agb 200
icesat2-boreal-v2.1-ht 200
icesat2-boreal-v3.0-agb 200
icesat2-boreal-v3.0-ht 200
Landsat8_SurfaceReflectance 200
NCEO_Africa_AGB_100m_2017 200
nisar-sim 200
Paraguay_Country_Pilot 200
SRTMGL1_COD 200
